# IPC Kenya Food Insecurity Data Cleaning

This notebook cleans the raw IPC Kenya food insecurity dataset downloaded from HDX.

Main tasks:
- Load the raw IPC CSV file
- Inspect columns, rows, and dates
- Standardize county names
- Filter to current IPC observations
- Remove non-target areas
- Aggregate sub-areas such as Marsabit and Turkana
- Create a modeling-ready county-level dataset
- Save the cleaned file to `02_data/processed/`

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
RAW_PATH = Path("../02_data/raw/ipc_ken_area_long.csv")
PROCESSED_PATH = Path("../02_data/processed/ipc_max_phase_per_county.csv")

print("Raw file exists:", RAW_PATH.exists())
print("Processed folder exists:", PROCESSED_PATH.parent.exists())

Raw file exists: True
Processed folder exists: True


In [4]:
df = pd.read_csv(RAW_PATH)

print("Rows and columns:", df.shape)
df.head()

Rows and columns: (5047, 11)


,Date of analysis,Country,Total country population,Level 1,Area,Validity period,From,To,Phase,Number,Percentage
0,Feb 2026,KEN,53331000,NaN,Baringo,current,2026-01-01,2026-03-31,all,764000,1.00
1,Feb 2026,KEN,53331000,NaN,Baringo,current,2026-01-01,2026-03-31,3+,114600,0.15
2,Feb 2026,KEN,53331000,NaN,Baringo,current,2026-01-01,2026-03-31,1,382000,0.50
3,Feb 2026,KEN,53331000,NaN,Baringo,current,2026-01-01,2026-03-31,2,267400,0.35
4,Feb 2026,KEN,53331000,NaN,Baringo,current,2026-01-01,2026-03-31,3,114600,0.15


In [5]:
df.columns.tolist()

['Date of analysis',
 'Country',
 'Total country population',
 'Level 1',
 'Area',
 'Validity period',
 'From',
 'To',
 'Phase',
 'Number',
 'Percentage']

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5047 entries, 0 to 5046
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Date of analysis          5047 non-null   str    
 1   Country                   5047 non-null   str    
 2   Total country population  5047 non-null   int64  
 3   Level 1                   2912 non-null   str    
 4   Area                      5047 non-null   str    
 5   Validity period           5047 non-null   str    
 6   From                      5047 non-null   str    
 7   To                        5047 non-null   str    
 8   Phase                     5047 non-null   str    
 9   Number                    5047 non-null   int64  
 10  Percentage                5047 non-null   float64
dtypes: float64(1), int64(2), str(8)
memory usage: 433.9 KB


In [7]:
df["Validity period"].value_counts()

Validity period
first projection    2604
current             2443
Name: count, dtype: int64

In [8]:
df["Date of analysis"].value_counts().sort_index()

Date of analysis
Apr 2022    161
Aug 2020    490
Feb 2020    322
Feb 2021    322
Feb 2022    322
Feb 2024    322
Feb 2025    322
Feb 2026    364
Jan 2023    392
Jul 2019    322
Jul 2021    322
Jul 2022    322
Jul 2023    322
Jul 2024    420
Jul 2025    322
Name: count, dtype: int64

In [9]:
sorted(df["Area"].dropna().unique())

['BANGLADESH',
 'Baringo',
 'DANDORA',
 'Dadaab',
 'Elgeyo-Marakwet',
 'Embu',
 'Embu (Mbeere)',
 'GITHURAI',
 'Garissa',
 'Homabay',
 'Isiolo',
 'KANGEMI',
 'KAWANGWARE',
 'KAYOLE',
 'KIBRA',
 'KONDELE',
 'Kajiado',
 'Kakuma',
 'Kalobeyei',
 'Kiambu',
 'Kilifi',
 'Kitui',
 'Kwale',
 'Laikipia',
 'Lamu',
 'Lamu county',
 'MATHARE',
 'MUKURU',
 'MWEMBE TAYARI',
 'Machakos',
 'Makueni',
 'Mandera',
 'Marsabit',
 'Marsabit - laisamis',
 'Marsabit - moyale',
 'Marsabit - north horr',
 'Marsabit - saku',
 'Meru',
 'Migori',
 'Narok',
 'Nyeri',
 'OBUNGA',
 'Samburu',
 'TANA RIVER',
 'Taita',
 'Taita Taveta',
 'Taita taveta',
 'Tana River',
 'Tana river',
 'Tharaka',
 'Tharaka Nithi',
 'Tharaka-nithi',
 'Turkana',
 'Turkana central',
 'Turkana east-kibish-loima',
 'Turkana north',
 'Turkana south',
 'Turkana west',
 'Wajir',
 'West Pokot',
 'West pokot']

## Clean County Names

The raw IPC dataset contains inconsistent area names. Some counties are written in different formats, while some county records are split into sub-areas. This step creates a clean `county` column for modeling.

In [10]:
def clean_area_name(area):
    """
    Standardize IPC area names into clean county names.
    This helps us combine records that refer to the same county
    but are written differently in the raw dataset.
    """
    
    mapping = {
        "TANA RIVER": "Tana River",
        "Tana river": "Tana River",
        "Taita": "Taita Taveta",
        "Taita taveta": "Taita Taveta",
        "Tharaka": "Tharaka Nithi",
        "Tharaka-nithi": "Tharaka Nithi",
        "West pokot": "West Pokot",
        "Lamu county": "Lamu",
        "Embu (Mbeere)": "Embu",
    }

    if area in mapping:
        return mapping[area]

    if str(area).startswith("Marsabit -"):
        return "Marsabit"

    if str(area).startswith("Turkana "):
        return "Turkana"

    return area


df["county"] = df["Area"].apply(clean_area_name)

df[["Area", "county"]].drop_duplicates().sort_values("county").head(40)

,Area,county
4319,BANGLADESH,BANGLADESH
0,Baringo,Baringo
4347,DANDORA,DANDORA
14,Dadaab,Dadaab
2436,Elgeyo-Marakwet,Elgeyo-Marakwet
4739,Embu (Mbeere),Embu
28,Embu,Embu
4361,GITHURAI,GITHURAI
42,Garissa,Garissa
2422,Homabay,Homabay


In [11]:
sorted(df["county"].dropna().unique())

['BANGLADESH',
 'Baringo',
 'DANDORA',
 'Dadaab',
 'Elgeyo-Marakwet',
 'Embu',
 'GITHURAI',
 'Garissa',
 'Homabay',
 'Isiolo',
 'KANGEMI',
 'KAWANGWARE',
 'KAYOLE',
 'KIBRA',
 'KONDELE',
 'Kajiado',
 'Kakuma',
 'Kalobeyei',
 'Kiambu',
 'Kilifi',
 'Kitui',
 'Kwale',
 'Laikipia',
 'Lamu',
 'MATHARE',
 'MUKURU',
 'MWEMBE TAYARI',
 'Machakos',
 'Makueni',
 'Mandera',
 'Marsabit',
 'Meru',
 'Migori',
 'Narok',
 'Nyeri',
 'OBUNGA',
 'Samburu',
 'Taita Taveta',
 'Tana River',
 'Tharaka Nithi',
 'Turkana',
 'Wajir',
 'West Pokot']

## Filter to Target ASAL Counties

The raw IPC file includes county records, urban settlements, refugee/camp areas, and Non-ASAL/Diaspora areas. For this project, we only keep the target county-level records needed for modeling.

In [12]:
target_counties = [
    "Baringo",
    "Embu",
    "Garissa",
    "Isiolo",
    "Kajiado",
    "Kilifi",
    "Kitui",
    "Kwale",
    "Laikipia",
    "Lamu",
    "Makueni",
    "Mandera",
    "Marsabit",
    "Meru",
    "Narok",
    "Nyeri",
    "Samburu",
    "Taita Taveta",
    "Tana River",
    "Tharaka Nithi",
    "Turkana",
    "Wajir",
    "West Pokot"
]

len(target_counties)

23

In [13]:
df["date"] = pd.to_datetime(df["Date of analysis"], format="%b %Y")
df["from_date"] = pd.to_datetime(df["From"])
df["to_date"] = pd.to_datetime(df["To"])

df[["Date of analysis", "date", "From", "from_date", "To", "to_date"]].head()

,Date of analysis,date,From,from_date,To,to_date
0,Feb 2026,2026-02-01,2026-01-01,2026-01-01,2026-03-31,2026-03-31
1,Feb 2026,2026-02-01,2026-01-01,2026-01-01,2026-03-31,2026-03-31
2,Feb 2026,2026-02-01,2026-01-01,2026-01-01,2026-03-31,2026-03-31
3,Feb 2026,2026-02-01,2026-01-01,2026-01-01,2026-03-31,2026-03-31
4,Feb 2026,2026-02-01,2026-01-01,2026-01-01,2026-03-31,2026-03-31


In [14]:
current_df = df[
    (df["Validity period"] == "current") &
    (df["county"].isin(target_counties))
].copy()

print("Rows after filtering:", current_df.shape)
current_df[["Date of analysis", "county", "Validity period", "Phase", "Number", "Percentage"]].head(10)

Rows after filtering: (2303, 15)


,Date of analysis,county,Validity period,Phase,Number,Percentage
0,Feb 2026,Baringo,current,all,764000,1.00
1,Feb 2026,Baringo,current,3+,114600,0.15
2,Feb 2026,Baringo,current,1,382000,0.50
3,Feb 2026,Baringo,current,2,267400,0.35
4,Feb 2026,Baringo,current,3,114600,0.15
5,Feb 2026,Baringo,current,4,0,0.00
6,Feb 2026,Baringo,current,5,0,0.00
28,Feb 2026,Embu,current,all,296000,1.00
29,Feb 2026,Embu,current,3+,14800,0.05
30,Feb 2026,Embu,current,1,103600,0.35


### Filtering Result

After filtering the raw IPC dataset to `current` records and the 23 target counties, the dataset was reduced from 5,047 rows to 2,303 rows. This removed projection records, urban settlements, refugee/camp areas, and other non-target analysis areas.

The filtered dataset is still in long format, meaning each county-date combination has multiple rows for IPC phases such as `all`, `3+`, `1`, `2`, `3`, `4`, and `5`. The next step is to aggregate this into one modeling-ready row per county per analysis period.

## Check County Coverage

Before creating the final modeling-ready dataset, we check how many target counties appear in each IPC analysis period. This helps us identify missing county records or unusual reporting periods.

In [15]:
county_coverage = (
    current_df[current_df["Phase"] == "all"]
    .groupby("Date of analysis")["county"]
    .nunique()
    .reset_index(name="county_count")
    .sort_values("Date of analysis")
)

county_coverage

,Date of analysis,county_count
0,Aug 2020,23
1,Feb 2020,23
2,Feb 2021,23
3,Feb 2022,23
4,Feb 2024,23
5,Feb 2025,23
6,Feb 2026,23
7,Jan 2023,23
8,Jul 2019,23
9,Jul 2021,23


In [16]:
coverage_detail = (
    current_df[current_df["Phase"] == "all"]
    .groupby(["Date of analysis", "county"])
    .size()
    .reset_index(name="records")
)

all_periods = sorted(current_df["Date of analysis"].unique())

for period in all_periods:
    counties_present = set(
        coverage_detail[coverage_detail["Date of analysis"] == period]["county"]
    )
    missing = sorted(set(target_counties) - counties_present)
    
    print(f"\n{period}")
    print(f"Counties present: {len(counties_present)}")
    print(f"Missing counties: {missing}")


Aug 2020
Counties present: 23
Missing counties: []

Feb 2020
Counties present: 23
Missing counties: []

Feb 2021
Counties present: 23
Missing counties: []

Feb 2022
Counties present: 23
Missing counties: []

Feb 2024
Counties present: 23
Missing counties: []

Feb 2025
Counties present: 23
Missing counties: []

Feb 2026
Counties present: 23
Missing counties: []

Jan 2023
Counties present: 23
Missing counties: []

Jul 2019
Counties present: 23
Missing counties: []

Jul 2021
Counties present: 23
Missing counties: []

Jul 2022
Counties present: 23
Missing counties: []

Jul 2023
Counties present: 23
Missing counties: []

Jul 2024
Counties present: 23
Missing counties: []

Jul 2025
Counties present: 23
Missing counties: []


## Create Modeling-Ready Target Variable

The raw IPC file has multiple phase rows per county and analysis period. For modeling, we create one row per county per period. The main target variable is `max_ipc_phase`, which shows the highest IPC phase with a population greater than zero.

In [17]:
phase_df = current_df[current_df["Phase"].isin(["1", "2", "3", "4", "5"])].copy()

phase_df["phase_num"] = phase_df["Phase"].astype(int)

phase_agg = (
    phase_df
    .groupby(["date", "Date of analysis", "county", "phase_num"], as_index=False)
    .agg(population_in_phase=("Number", "sum"))
)

max_phase = (
    phase_agg[phase_agg["population_in_phase"] > 0]
    .groupby(["date", "Date of analysis", "county"], as_index=False)["phase_num"]
    .max()
    .rename(columns={"phase_num": "max_ipc_phase"})
)

max_phase.head()

,date,Date of analysis,county,max_ipc_phase
0,2019-07-01,Jul 2019,Baringo,4
1,2019-07-01,Jul 2019,Embu,3
2,2019-07-01,Jul 2019,Garissa,4
3,2019-07-01,Jul 2019,Isiolo,4
4,2019-07-01,Jul 2019,Kajiado,3


### County Coverage and Target Variable Result

After filtering to `current` records and the 23 target counties, every available current IPC analysis period contains all 23 counties. This creates a balanced county-period dataset for the first modeling-ready output.

The `max_ipc_phase` variable was created by selecting the highest IPC phase with a population greater than zero for each county and analysis period. This gives one severity score per county-period and will be used as the main target variable for modeling.

## Add Population Metrics

The `max_ipc_phase` column gives the highest IPC severity level for each county and analysis period. We now add population metrics so the final dataset shows both severity and scale.

The added columns are:
- `total_population`: total county population in the IPC analysis
- `phase_3_plus_population`: number of people in Crisis level or worse
- `phase_3_plus_percentage`: share of the population in Crisis level or worse

In [18]:
total_population = (
    current_df[current_df["Phase"] == "all"]
    .groupby(["date", "Date of analysis", "county"], as_index=False)["Number"]
    .sum()
    .rename(columns={"Number": "total_population"})
)

phase_3_plus = (
    current_df[current_df["Phase"] == "3+"]
    .groupby(["date", "Date of analysis", "county"], as_index=False)["Number"]
    .sum()
    .rename(columns={"Number": "phase_3_plus_population"})
)

total_population.head()

,date,Date of analysis,county,total_population
0,2019-07-01,Jul 2019,Baringo,703697
1,2019-07-01,Jul 2019,Embu,219220
2,2019-07-01,Jul 2019,Garissa,431950
3,2019-07-01,Jul 2019,Isiolo,155465
4,2019-07-01,Jul 2019,Kajiado,870721


In [19]:
processed_df = (
    max_phase
    .merge(total_population, on=["date", "Date of analysis", "county"], how="left")
    .merge(phase_3_plus, on=["date", "Date of analysis", "county"], how="left")
)

processed_df["phase_3_plus_percentage"] = (
    processed_df["phase_3_plus_population"] / processed_df["total_population"]
).round(4)

processed_df = processed_df.rename(columns={
    "Date of analysis": "analysis_period"
})

processed_df = processed_df[
    [
        "date",
        "analysis_period",
        "county",
        "max_ipc_phase",
        "phase_3_plus_population",
        "total_population",
        "phase_3_plus_percentage"
    ]
]

processed_df.head(10)

,date,analysis_period,county,max_ipc_phase,phase_3_plus_population,total_population,phase_3_plus_percentage
0,2019-07-01,Jul 2019,Baringo,4,105555,703697,0.15
1,2019-07-01,Jul 2019,Embu,3,32883,219220,0.15
2,2019-07-01,Jul 2019,Garissa,4,151183,431950,0.35
3,2019-07-01,Jul 2019,Isiolo,4,54413,155465,0.35
4,2019-07-01,Jul 2019,Kajiado,3,43536,870721,0.05
5,2019-07-01,Jul 2019,Kilifi,3,209996,1399975,0.15
6,2019-07-01,Jul 2019,Kitui,3,219537,1097687,0.20
7,2019-07-01,Jul 2019,Kwale,3,123030,820199,0.15
8,2019-07-01,Jul 2019,Laikipia,3,50571,505712,0.10
9,2019-07-01,Jul 2019,Lamu,3,25629,128144,0.20


## Final Processed IPC Dataset Preview

The table above shows the first rows of the cleaned IPC dataset.

At this stage, the raw long-format IPC data has been converted into a modeling-ready table where each row represents one county in one IPC analysis period.

For each county-period, the dataset now includes:
- `max_ipc_phase`: the highest IPC phase where population was greater than zero
- `phase_3_plus_population`: number of people in IPC Phase 3 or worse
- `total_population`: total analyzed population for the county
- `phase_3_plus_percentage`: share of the county population in Crisis level or worse

This structure makes it easier to compare counties by both severity and population impact.

In [20]:
print("Processed shape:", processed_df.shape)

processed_df.groupby("analysis_period")["county"].nunique()

Processed shape: (322, 7)


analysis_period
Aug 2020    23
Feb 2020    23
Feb 2021    23
Feb 2022    23
Feb 2024    23
Feb 2025    23
Feb 2026    23
Jan 2023    23
Jul 2019    23
Jul 2021    23
Jul 2022    23
Jul 2023    23
Jul 2024    23
Jul 2025    23
Name: county, dtype: int64

## Final Dataset Validation

The final processed dataset contains 322 rows and 7 columns. This matches the expected structure of 14 current IPC analysis periods across 23 target counties.

Each analysis period contains all 23 counties, meaning the processed dataset is balanced and ready for exploratory analysis.

This file will be saved as `ipc_max_phase_per_county.csv` and used for the next project stage: exploratory data analysis.

In [21]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

processed_df.to_csv(PROCESSED_PATH, index=False)

print(f"Saved cleaned file to: {PROCESSED_PATH}")

Saved cleaned file to: ..\02_data\processed\ipc_max_phase_per_county.csv


In [22]:
PROCESSED_PATH.exists()

True